## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: YOLOv1
*****

Basada en la implementación: [Saptarshi MT
](https://medium.com/@saptarshimt/yolo-v1-pascal-voc-simplistic-pytorch-implementation-from-scratch-961fa36f4d4d)

In [ ]:
#!python -m pip install kagglehub

## Librerias

In [ ]:
import kagglehub
from random import sample
from tqdm import tqdm
from time import time
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import numpy as np
import pandas as pd
import  xml.etree.ElementTree as ET
import shutil
import torch
import os

from torch.utils.data import TensorDataset, DataLoader
import torchvision.transforms as transforms
import torch.nn as nn
import torchvision.models as models


## Funciones personalizadas

In [ ]:
# This function preprocesses an XML file containing object detection annotations.
def preprocess_xml(filename):

    # Parse the XML file using ElementTree.
    tree = ET.parse(filename)
    root = tree.getroot()

    # Extract image size from the XML.
    size_tree = root.find('size')
    height = float(size_tree.find('height').text)
    width = float(size_tree.find('width').text)

    # Initialize list to store bounding boxes.
    bounding_boxes = []

    # Create a dictionary to map class names to indices.
    class_dict = {classes[i]: i for i in range(len(classes))}

    # Iterate through each object annotation in the XML.
    for object_tree in root.findall('object'):
        # Extract bounding box coordinates.
        for bounding_box in object_tree.iter('bndbox'):
            xmin = float(bounding_box.find('xmin').text)
            xmax = float(bounding_box.find('xmax').text)
            ymin = float(bounding_box.find('ymin').text)
            ymax = float(bounding_box.find('ymax').text)
            break

        # Extract class name of the object.
        class_name = object_tree.find('name').text

        # Calculate normalized bounding box coordinates and append to the list.
        bounding_box = [
            (xmin + xmax) / (2 * width),
            (ymin + ymax) / (2 * height),
            (xmax - xmin) / width,
            (ymax - ymin) / height,
            class_dict[class_name]
        ]
        bounding_boxes.append(bounding_box)

    # Convert the list of bounding boxes to a tensor and return.
    return torch.tensor(bounding_boxes)

In [ ]:
def generate_output(bounding_boxes):
    # Initialize output label tensor
    output_label = np.zeros((SPLIT_SIZE, SPLIT_SIZE, N_CLASSES + 5))

    # Iterate through bounding boxes
    for b in range(len(bounding_boxes)):
        # Calculate grid positions
        grid_x = bounding_boxes[..., b, 0] * SPLIT_SIZE
        grid_y = bounding_boxes[..., b, 1] * SPLIT_SIZE

        # Convert to integer grid indices
        i = int(grid_x)
        j = int(grid_y)

        # Assign values to output label tensor
        output_label[i, j, 0:5] = [1., int(grid_x), int(grid_y), bounding_boxes[..., b, 2], bounding_boxes[..., b, 3]]
        output_label[i, j, 5 + int(bounding_boxes[..., b, 4])] = 1.

    # Convert output label tensor to TensorFlow tensor
    return torch.tensor(output_label)

## Diseño del modelo

#### Funcion de perdida

In [ ]:
def difference(x, y):
    return torch.sum((y - x)**2)

def yolo_loss(y_true, y_pred):
    target = y_true[..., 0]  # Grid cell where we have information of object presence

    #--------------------------------------- for object

    target_indices = target == 1
    y_pred_extract = y_pred[target_indices]
    y_target_extract = y_true[target_indices]

    object_loss = difference(y_pred_extract[..., 0].float(), torch.ones_like(y_pred_extract[..., 0]).float())

    #------------------------------------------------------ for no object

    target_indices_no_obj = target == 0
    y_pred_extract_no_obj = y_pred[target_indices_no_obj]
    y_target_extract_no_obj = y_true[target_indices_no_obj]

    no_obj_loss = difference(y_pred_extract_no_obj[..., 0].float(), torch.zeros_like(y_pred_extract_no_obj[..., 0]).float())

    #-------------------------------------------------------- for object class loss

    y_pred_extract_class = y_pred[target_indices][:, 5:]
    class_extract = y_true[target_indices][:, 5:]

    class_loss = difference(y_pred_extract_class.float(), class_extract.float())

    #--------------------------------------------------------- for object center loss

    y_pred_extract_center = y_pred[target_indices][:, 1:3]
    center_target = y_true[target_indices][:, 1:3]

    center_loss = difference(y_pred_extract_center.float(), center_target.float())

    #------------------------------------------------------- for width and height

    size_pred = y_pred[target_indices][:, 3:5]
    size_target = y_true[target_indices][:, 3:5]

    size_loss = difference(torch.sqrt(torch.abs(size_pred.float())), torch.sqrt(torch.abs(size_target.float())))

    box_loss = center_loss + size_loss

    lambda_coord = 5
    lambda_no_obj = 0.5

    loss = object_loss + (lambda_no_obj * no_obj_loss) + lambda_coord * box_loss + class_loss

    return loss

## Dataset

<center>
    <img src=https://viso.ai/wp-content/uploads/2024/01/Results-of-localization-on-PASCAL-VOC-dataset-Green-box-Estimated-Window-Red-box-Ground-Truth.jpg width=800>
</center>

El conjunto de datos [PASCAL VOC (Visual Object Classes)](http://host.robots.ox.ac.uk/pascal/VOC/) es un conocido conjunto de datos de detección de objetos, segmentación y clasificación. Está diseñado para fomentar la investigación sobre una amplia variedad de categorías de objetos y se utiliza comúnmente para evaluar modelos de visión artificial. Es un conjunto de datos esencial para los investigadores y desarrolladores que trabajan en tareas de detección de objetos, segmentación y clasificación.



La data incluye las siguientes clases:

* __Persona__: persona
* __Animal__: pájaro, gato, vaca, perro, caballo, oveja
* __Vehículo__: avión, bicicleta, barco, autobús, coche, moto, tren
* __Interior__: botella, silla, mesa de comedor, planta en maceta, sofá, televisor/monitor

#### Descarga de datos

In [ ]:
path = kagglehub.dataset_download("vijayabhaskar96/pascal-voc-2007-and-2012")
print("Path to dataset files:", path)

#### Carga de datos y preprocesamiento

In [ ]:
train_images=f"{path}/" + "VOCdevkit/VOC2007/JPEGImages/"
train_maps=f"{path}/" + "VOCdevkit/VOC2007/Annotations/"

classes=['aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair','cow',
         'diningtable','dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor']

N_CLASSES=len(classes)
H,W=224,224
SPLIT_SIZE=int(H/32)
BATCH_SIZE=32

In [ ]:
im_paths=[]
xml_paths=[]

files_list = os.listdir(train_maps)
files_list = sample(files_list, 5000)

print('Total images: {}'.format(len(files_list)))

for i in files_list:
    im_paths.append(train_images+i[:-3]+'jpg')
    xml_paths.append(train_maps+i)

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
])

images = []
for im_path in tqdm(im_paths):
    # Open image using PIL
    img = Image.open(im_path).convert('RGB')

    # Apply transformations
    img_tensor = transform(img)

    images.append(img_tensor)

# Stack tensors
images = torch.stack(images)

In [ ]:
boxes = []

for xml_path in tqdm(xml_paths):
    boxes.append(generate_output(preprocess_xml(xml_path)))

boxes = torch.stack(boxes)

In [ ]:
boxes.shape

## Modelo

In [ ]:
def ConvBlock(in_ch: int, out_ch: int, k: int = 3, s: int = 1, p = None):

  if p is None:
      p = (k - 1) // 2

  model = nn.Sequential(
    nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
    nn.BatchNorm2d(out_ch),
    nn.LeakyReLU(0.1, inplace=True)
  )
  return model

In [ ]:
## Base model
input_ch = 3
out_ch = 2048

C = []
# 224 -> 112
C += [ConvBlock(input_ch, 64, 7, 2, 3),
      nn.MaxPool2d(2, 2)]

# 112 -> 56
C += [ConvBlock(64, 192, 3, 1),
      nn.MaxPool2d(2, 2)]

# 56 -> 28
C += [ConvBlock(192, 128, 1, 1),
      ConvBlock(128, 256, 3, 1),
      ConvBlock(256, 256, 1, 1),
      ConvBlock(256, 512, 3, 1),
      nn.MaxPool2d(2, 2)]

# 28 -> 14
# 4 bloques conv pares
for _ in range(4):
    C += [ConvBlock(512, 256, 1, 1),
          ConvBlock(256, 512, 3, 1)]
C += [ConvBlock(512, 512, 1, 1),
      ConvBlock(512, 1024, 3, 1),
      nn.MaxPool2d(2, 2)]

# 14 -> 7
C += [ConvBlock(1024, 512, 1, 1),
      ConvBlock(512, 1024, 3, 1),
      ConvBlock(1024, 1024, 3, 1),
      ConvBlock(1024, 1024, 3, 2)]

# ahora 7x7 -> 4x4
C += [ConvBlock(1024, out_ch, 3, 1)]
base_model = nn.Sequential(*C)

#print(base_model)

In [ ]:
NUM_FILTERS = 512
OUTPUT_DIM = int(N_CLASSES + 5)

flatten = nn.Flatten()

dense_layers = nn.Sequential(
    nn.Linear(out_ch * 4 * 4, NUM_FILTERS),
    nn.BatchNorm1d(NUM_FILTERS),
    nn.LeakyReLU(0.1),

    nn.Linear(NUM_FILTERS, int(SPLIT_SIZE * SPLIT_SIZE * OUTPUT_DIM)),
    nn.Sigmoid()
)

# Combine all layers into a single sequential model
model = nn.Sequential(
    base_model,
    #conv_layers,
    flatten,
    dense_layers,
    nn.Unflatten(1, (SPLIT_SIZE, SPLIT_SIZE, OUTPUT_DIM))
)

# Print model summary
#print(model)

In [ ]:
model = nn.Sequential(
    base_model,
    flatten,
    dense_layers,
    nn.Unflatten(1, (SPLIT_SIZE, SPLIT_SIZE, OUTPUT_DIM))
)

#### Entrenamiento del modelo

In [ ]:
## Tiempo estimado de ejecución: 48 minutos (GPU)
start = time()

# Move model and data to MPS device
device = "cuda" if torch.cuda.is_available() else "cpu"
device
print(f'(device) {device}')

model.to(device)
images = images.to(device)
boxes = boxes.to(device)


# Define your loss function and optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # Assuming batch size of 32 and iterating through batches
    batch_size = 32
    for i in range(0, len(images), batch_size):
        img_batch = images[i:i+batch_size]
        target_batch = boxes[i:i+batch_size]

        #print('(shape-batch) img: {}, target: {}'.format(img_batch.shape, target_batch.shape))
        # Forward pass
        outputs = model(img_batch)

        # Assuming you have a custom YOLO-like loss function
        loss = yolo_loss(target_batch,outputs)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Print average loss for epoch
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / (len(images) / batch_size)}')

print('Finished Training')
print('Time spent[s]: {:.4f}'.format(time()-start))

In [ ]:
## Guardar el modelo
#torch.save(model, 'YOLOv1.pth')

Modelo entrenado [descarga](https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/Edt5XYpIbixDpxf_6AaCdF4BT77BPVgCimhtYkMzdDg_uw?e=JAjgKQ&download=1)

In [ ]:
if os.name == 'posix':

    ## Descarga del modelo entrenado
    !wget "https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/Edt5XYpIbixDpxf_6AaCdF4BT77BPVgCimhtYkMzdDg_uw?e=JAjgKQ&download=1"
    !mv "Edt5XYpIbixDpxf_6AaCdF4BT77BPVgCimhtYkMzdDg_uw?e=JAjgKQ&download=1" "YOLOv1.pth"

## Cargar el modelo entrenado
#model = torch.load('YOLOv1.pth', weights_only=False)

#### Ilustración de detección

In [ ]:
for j in [60,10,5,4,3416,500,600,800,560,1000,595,1400,1657]:

    output = model(images[j:j+32])[:1] #SINCE MODEL TAKES 32 AS BATCH AND NOT SINGLE IMAGE

    # Threshold for detection
    THRESH = 0.25

    # Get object positions
    object_positions = torch.where(output[..., 0] >= THRESH)
    selected_output = output[object_positions]

    # Initialize lists for final boxes and scores
    final_boxes = []
    final_scores = []

    # Iterate through object positions
    for i in range(len(object_positions[0])):
        if selected_output[i][0] > THRESH:
            # Extract box parameters
            output_box = selected_output[i][1:5].float()
            x_centre = (object_positions[1][i] + output_box[0]) * 32
            y_centre = (object_positions[2][i] + output_box[1]) * 32
            x_width, y_height = torch.abs(224 * output_box[2]), torch.abs(224 * output_box[3])
            x_min, y_min = int(x_centre - (x_width / 2)), int(y_centre - (y_height / 2))
            x_max, y_max = int(x_centre + (x_width / 2)), int(y_centre + (y_height / 2))

            # Adjust bounding box coordinates
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(224, x_max)
            y_max = min(224, y_max)

            # Append to final boxes
            final_boxes.append([x_min, y_min, x_max, y_max,
                               classes[torch.argmax(selected_output[i][5:]).item()]])
            final_scores.append(selected_output[i][0].item())


    img = np.array(images[j].permute(1, 2, 0).to('cpu')).copy()

    for i in range(len(final_boxes)):

        # Extract object classes and boxes
        object_classes = final_boxes[i][4]
        nms_output = final_boxes[i][0:4]

        x1, y1 = nms_output[:2]
        x2, y2 = nms_output[2:]

        # Specify the color (in BGR format) and thickness
        color = (255, 0, 0)
        thickness = 2

        # Draw the rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

        # Add text (object classes)
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        font_color = (255, 255, 255)
        text_position = (x1, y2)

        cv2.putText(img, object_classes, text_position, font, font_scale, font_color, thickness)
    plt.imshow(img)
    plt.show()